# Session 9: Wireless Technologies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/03-computer-networks/code/session-09-wireless-technologies.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-09_wifi_planner.py`

In [1]:
"""Session 9 - Wi-Fi planning maths: 2.4 GHz channel overlap, free-space path
loss and a simple link budget for an access point.

FSPL(dB) = 20 log10(d_km) + 20 log10(f_MHz) + 32.44
Received power = Tx power + Tx antenna gain + Rx antenna gain - FSPL - wall losses
Run:  python session-09_wifi_planner.py
"""
import math


def channel_centre_24(ch):           # 2.4 GHz channels are 5 MHz apart
    return 2407 + 5 * ch


def overlaps(c1, c2, width=22):        # classic 802.11b/g channel is 22 MHz wide
    return abs(channel_centre_24(c1) - channel_centre_24(c2)) < width


def fspl(d_m, f_mhz):
    return 20 * math.log10(d_m / 1000) + 20 * math.log10(f_mhz) + 32.44


def rssi(d_m, f_mhz, tx_dbm=20, gain_tx=3, gain_rx=0, walls=0, wall_db=5):
    return tx_dbm + gain_tx + gain_rx - fspl(d_m, f_mhz) - walls * wall_db


def quality(dbm):
    if dbm >= -60: return "excellent"
    if dbm >= -67: return "good (VoIP OK)"
    if dbm >= -75: return "fair (browsing)"
    return "poor"


if __name__ == "__main__":
    print("2.4 GHz channels that do NOT overlap with channel 1:",
          [c for c in range(1, 14) if not overlaps(1, c)][:3], "-> use 1, 6, 11")
    print("Channel 3 overlaps channel 1?", overlaps(1, 3), "| channel 6?", overlaps(1, 6))

    print("\nSignal at distance from a 20 dBm AP with a 3 dBi antenna:")
    print(f"{'distance':>8} {'walls':>5} {'2.4 GHz':>9} {'5 GHz':>9}  5 GHz quality")
    for d, w in [(5, 0), (15, 1), (25, 2), (40, 2), (60, 3)]:
        a, b = rssi(d, 2437, walls=w), rssi(d, 5500, walls=w)
        print(f"{d:>6} m {w:>5} {a:>7.1f}dBm {b:>7.1f}dBm  {quality(b)}")
    print("\n5 GHz loses about", round(fspl(10, 5500) - fspl(10, 2437), 1),
          "dB more than 2.4 GHz at the same distance (shorter range, more channels).")

2.4 GHz channels that do NOT overlap with channel 1: [6, 7, 8] -> use 1, 6, 11
Channel 3 overlaps channel 1? True | channel 6? False

Signal at distance from a 20 dBm AP with a 3 dBi antenna:
distance walls   2.4 GHz     5 GHz  5 GHz quality
     5 m     0   -31.2dBm   -38.2dBm  excellent
    15 m     1   -45.7dBm   -52.8dBm  excellent
    25 m     2   -55.1dBm   -62.2dBm  good (VoIP OK)
    40 m     2   -59.2dBm   -66.3dBm  good (VoIP OK)
    60 m     3   -67.7dBm   -74.8dBm  fair (browsing)

5 GHz loses about 7.1 dB more than 2.4 GHz at the same distance (shorter range, more channels).


## `session-09_wpa2_psk.py`

In [2]:
"""Session 9 - Why WPA2-Personal needs a long passphrase.

WPA2-PSK turns the passphrase into a 256-bit Pairwise Master Key with
PBKDF2-HMAC-SHA1(passphrase, SSID, 4096 iterations, 32 bytes).
The SSID is the salt, so the same password on a different network gives a
different key. Anyone who records a handshake can try guesses OFFLINE, so the
only defence for PSK networks is a long, random passphrase (or WPA3-SAE).
Run:  python session-09_wpa2_psk.py
"""
import hashlib


def pmk(passphrase: str, ssid: str) -> bytes:
    return hashlib.pbkdf2_hmac("sha1", passphrase.encode(), ssid.encode(), 4096, 32)


if __name__ == "__main__":
    print("PMK for 'CampusLab2026!' on SSID 'LAB-WIFI':")
    print(" ", pmk("CampusLab2026!", "LAB-WIFI").hex())
    print("Same passphrase, SSID 'HOME-WIFI':")
    print(" ", pmk("CampusLab2026!", "HOME-WIFI").hex(), "(different: SSID is the salt)")

    # Each guess costs 4096 x 2 SHA-1 operations. Assume an attacker's GPU manages
    # about one million guesses per second (order-of-magnitude teaching figure).
    gpu = 1_000_000
    print()
    print(f"{'passphrase style':<34}{'combinations':>14}  worst-case time on 1 GPU")
    styles = [("8 lowercase letters", 26 ** 8), ("8 chars, upper+lower+digits", 62 ** 8),
              ("10 chars, 94 printable symbols", 94 ** 10), ("4 random dictionary words (7776)", 7776 ** 4),
              ("6 random dictionary words", 7776 ** 6)]
    for name, space in styles:
        secs = space / gpu
        if secs < 86400: t = f"{secs / 3600:,.1f} hours"
        elif secs < 3.15e7: t = f"{secs / 86400:,.1f} days"
        elif secs < 3.15e7 * 1000: t = f"{secs / 3.15e7:,.1f} years"
        else: t = f"{secs / 3.15e7:.1e} years"
        print(f"{name:<34}{space:>14.2e}  {t}")

PMK for 'CampusLab2026!' on SSID 'LAB-WIFI':
  602f02ceb0f8d67af471def10a6b8cee6a9fd518c68f2cf1b62edef93ea7af78
Same passphrase, SSID 'HOME-WIFI':
  95d3e85ef7ebc0c449cb63cde980a80b6465f8e9348a7686f9e7d789b4ac8a91 (different: SSID is the salt)

passphrase style                    combinations  worst-case time on 1 GPU
8 lowercase letters                     2.09e+11  2.4 days
8 chars, upper+lower+digits             2.18e+14  6.9 years
10 chars, 94 printable symbols          5.39e+19  1.7e+06 years
4 random dictionary words (7776)        3.66e+15  116.1 years
6 random dictionary words               2.21e+23  7.0e+09 years
